# Análise do threshold superior adaptativo

Compara P99.9, P99.7 e P99.5 no conjunto de validação e investiga os exames
com maior alteração de Dice. Para esses casos, o notebook também pode calcular
o valor efetivo dos percentis em HU após o mesmo downsampling do pipeline.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from scipy.stats import wilcoxon

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = next(
    root for root in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (root / "src" / "utils").is_dir()
)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils.experiments import (  # noqa: E402
    build_mean_normalized_intensity_histogram,
    build_normalized_intensity_histograms,
    build_threshold_performance_data,
    compute_effective_upper_thresholds,
    compute_intensity_histogram_analysis,
    load_parameter_validation_run,
)
from utils.project.notebook_env import resolve_imagecas_base_path  # noqa: E402
from utils.visualization import (  # noqa: E402
    calculate_binned_intensity_mean_median,
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)

## Configuração

Os dois runs são fixados abaixo para manter a análise reproduzível.
`LOAD_IMAGE_DATA=False` executa somente as análises baseadas nos CSVs; quando
ativado, o notebook também calcula thresholds HU e os histogramas de toda a
coorte de validação. Essa etapa é mais demorada porque processa as 270 imagens.

In [ ]:
RUNS_DIR = (
    REPO_ROOT
    / "output/segmentation/analysis/pipeline_parameter_validation/runs"
)
PRIMARY_RUN_NAME = "sensitivity_cbeb_p999_val_270"
COMPARISON_RUN_NAME = "sensitivity_selected_val_270"

DETAIL_CASE_COUNT = 30
LOAD_IMAGE_DATA = True
DENSE_INTENSITY_MIN_HU = 300.0
HISTOGRAM_BINS = 256
HISTOGRAM_PROGRESS_EVERY = 25

try:
    IMAGECAS_PATH = resolve_imagecas_base_path() if LOAD_IMAGE_DATA else None
except FileNotFoundError:
    IMAGECAS_PATH = None

In [ ]:
# Carrega o run P99.9 e o run histórico que contém P99.7/P99.5.
RUN_DIR, primary_results, primary_parameters, run_config = (
    load_parameter_validation_run(RUNS_DIR / PRIMARY_RUN_NAME)
)
THRESHOLD_RUN_DIR, threshold_results, threshold_parameters, _ = (
    load_parameter_validation_run(RUNS_DIR / COMPARISON_RUN_NAME)
)

# No run histórico, o baseline representa P99.7; P99.9 vem do run principal.
threshold_aliases = {"baseline": "upper_p997", "upper_p995": "upper_p995"}
threshold_results = threshold_results.loc[
    threshold_results["variant"].isin(threshold_aliases)
].copy()
threshold_results["variant"] = threshold_results["variant"].replace(
    threshold_aliases
)

threshold_parameters = threshold_parameters.loc[
    threshold_parameters["variant"].isin(threshold_aliases)
].copy()
threshold_parameters["variant"] = threshold_parameters["variant"].replace(
    threshold_aliases
)
threshold_parameters.loc[
    threshold_parameters["variant"].eq("upper_p997"), "parameter_group"
] = "upper_percentile"
threshold_parameters.loc[
    threshold_parameters["variant"].eq("upper_p997"), "description"
] = "Percentil superior do threshold = 99.7."

results_df = pd.concat([primary_results, threshold_results], ignore_index=True)
parameters_df = pd.concat(
    [primary_parameters, threshold_parameters], ignore_index=True
)
results_df.drop(
    columns=["cohort_kind", "cohort_roles"], errors="ignore", inplace=True
)

if set(primary_results["IMG_ID"]) != set(threshold_results["IMG_ID"]):
    raise ValueError("Os runs P99.9, P99.7 e P99.5 não usam os mesmos exames.")

print(f"Run principal: {RUN_DIR.relative_to(REPO_ROOT)}")
print(f"Run dos thresholds: {THRESHOLD_RUN_DIR.relative_to(REPO_ROOT)}")
print(
    f"Imagens: {results_df['IMG_ID'].nunique()} | "
    f"Variantes: {results_df['variant'].nunique()}"
)

## Comparação dos thresholds nos casos de maior variação

Esta seção usa P99.9 como baseline e, para cada exame, escolhe a melhor
alternativa entre P99.7 e P99.5. A análise detalhada considera os exames com
maior variação absoluta de Dice, independentemente de melhora ou piora.

### 1. Desempenho global por percentil

Compara P99.9, P99.7 e P99.5 considerando todos os 270 exames. A tabela mostra
Dice médio e mediano, sucesso dos óstios e a fração média do volume preservada
pelo threshold. P99.9 é a configuração de referência.

In [ ]:
THRESHOLD_VARIANTS = ("baseline", "upper_p997", "upper_p995")
THRESHOLD_LABELS = {
    "baseline": "P99.9 (baseline)",
    "upper_p997": "P99.7",
    "upper_p995": "P99.5",
}

# Organiza os três percentis usando os mesmos exames de validação.
threshold_performance_df = build_threshold_performance_data(
    results_df,
    parameters_df,
    variants=THRESHOLD_VARIANTS,
)
overall_threshold_summary = (
    threshold_performance_df.groupby(["variant", "upper_percentile"], as_index=False)
    .agg(
        images=("IMG_ID", "nunique"),
        mean_dice=("dice_artery", "mean"),
        median_dice=("dice_artery", "median"),
        ostia_success_rate=("ostia_success", "mean"),
        mean_threshold_volume_fraction=("threshold_volume_fraction", "mean"),
    )
)
overall_threshold_summary["ostia_success_percent"] = (
    100 * overall_threshold_summary["ostia_success_rate"]
)
overall_threshold_summary = overall_threshold_summary.sort_values(
    "upper_percentile", ascending=False
)
overall_threshold_summary["variant"] = overall_threshold_summary["variant"].map(
    THRESHOLD_LABELS
)
display(
    overall_threshold_summary.drop(columns="ostia_success_rate")
    .reset_index(drop=True)
    .round(4)
)

### 2. Seleção dos exames mais sensíveis

Para cada exame, P99.9 é comparado com a melhor alternativa entre P99.7 e
P99.5. Em seguida, são selecionados os exames com maior variação absoluta de
Dice. Portanto, entram tanto grandes melhorias quanto grandes pioras.

In [ ]:
# Compara o baseline com a melhor alternativa individual de cada exame.
threshold_dice_wide = threshold_performance_df.pivot(
    index="IMG_ID", columns="variant", values="dice_artery"
)
alternative_dice = threshold_dice_wide[["upper_p997", "upper_p995"]]

threshold_delta_df = pd.DataFrame(index=threshold_dice_wide.index)
threshold_delta_df["baseline_dice"] = threshold_dice_wide["baseline"]
threshold_delta_df["best_threshold_variant"] = alternative_dice.idxmax(axis=1)
threshold_delta_df["best_threshold_dice"] = alternative_dice.max(axis=1)
threshold_delta_df["delta_dice"] = (
    threshold_delta_df["best_threshold_dice"] - threshold_delta_df["baseline_dice"]
)
threshold_delta_df["absolute_delta_dice"] = threshold_delta_df["delta_dice"].abs()
threshold_delta_df = threshold_delta_df.reset_index()

# Selecionar as N maiores variações
largest_threshold_changes = threshold_delta_df.nlargest(
    DETAIL_CASE_COUNT, "absolute_delta_dice"
).copy()
largest_threshold_changes[
    ["IMG_ID", "baseline_dice", "best_threshold_variant", "best_threshold_dice", "delta_dice"]
].round(4)

cohort_image_ids = [int(image_id) for image_id in threshold_dice_wide.index]

### 3. Significância estatística da diferença de Dice

Wilcoxon bilateral pareado por IMG_ID, usando toda a validação, não somente os
casos selecionados acima. Delta positivo favorece o percentil alternativo.
O teste avalia os postos das diferenças sob a hipótese de simetria em torno de
zero; não testa diretamente a diferença das médias. A interpretação como
mudança de localização pressupõe diferenças aproximadamente simétricas e exames
independentes. Esta análise de validação é exploratória.

As diferenças são arredondadas a 12 casas para evitar desempates numéricos.
Pares sem Dice são excluídos e contabilizados; Dice zero (falha) é mantido.
Diferenças zero não entram nos postos (`zero_method="wilcox"`). Para pares todos
idênticos, convenciona-se p=1. Holm controla as duas comparações a 5%.

Referência: [Wilcoxon no SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.wilcoxon.html).


In [ ]:
wilcoxon_rows = []

for variant in ("upper_p997", "upper_p995"):
    paired_dice = threshold_dice_wide[["baseline", variant]].dropna()
    if paired_dice.empty:
        raise ValueError(f"Sem pares válidos para {variant}.")
    if not paired_dice.ge(0).all().all() or not paired_dice.le(1).all().all():
        raise ValueError("Dice deve ser finito e estar entre 0 e 1.")
    delta = paired_dice[variant] - paired_dice["baseline"]
    # Evita desempates artificiais por arredondamento em ponto flutuante.
    ranked_delta = delta.round(12)

    if ranked_delta.ne(0).any():
        statistic, p_value = wilcoxon(
            ranked_delta,
            alternative="two-sided",
            zero_method="wilcox",
            method="auto",
        )
    else:
        statistic, p_value = 0.0, 1.0

    wilcoxon_rows.append(
        {
            "comparison": f"P99.9 vs {THRESHOLD_LABELS[variant]}",
            "paired_images": len(delta),
            "excluded_pairs": len(threshold_dice_wide) - len(delta),
            "nonzero_pairs": int(ranked_delta.ne(0).sum()),
            "median_delta_dice": delta.median(),
            "mean_delta_dice": delta.mean(),
            "improved_images": int(ranked_delta.gt(0).sum()),
            "worse_images": int(ranked_delta.lt(0).sum()),
            "unchanged_images": int(ranked_delta.eq(0).sum()),
            "wilcoxon_statistic": statistic,
            "p_value": p_value,
        }
    )

wilcoxon_summary = pd.DataFrame(wilcoxon_rows)
# Holm controla a família das duas comparações e preserva a ordem da tabela.
ordered_p = wilcoxon_summary["p_value"].sort_values()
holm_factors = pd.Series(range(len(ordered_p), 0, -1), index=ordered_p.index)
wilcoxon_summary["p_value_holm"] = (
    ordered_p.mul(holm_factors).cummax().clip(upper=1)
)
wilcoxon_summary["significant_holm_005"] = wilcoxon_summary["p_value_holm"] < 0.05
# Não arredonda p pequeno para zero e não exige o Styler/Jinja2.
with pd.option_context("display.float_format", "{:.6g}".format):
    display(wilcoxon_summary)

### 4. Threshold efetivo em HU

O percentil configurado não corresponde a um valor HU fixo. Esta etapa carrega
somente os exames selecionados, aplica o mesmo downsampling do pipeline e
calcula os valores efetivos de P99.5, P99.7 e P99.9 em HU.

In [ ]:
# Calcula os valores HU somente para os exames de maior variação.
effective_thresholds_df = pd.DataFrame()
if LOAD_IMAGE_DATA:
    if IMAGECAS_PATH is None:
        print("Defina IMAGECAS_BASE_PATH para calcular os thresholds efetivos em HU.")
    else:
        effective_thresholds_df = compute_effective_upper_thresholds(
            largest_threshold_changes["IMG_ID"].astype(int).tolist(),
            IMAGECAS_PATH,
            run_config["effective_base_config"],
            percentiles=(99.5, 99.7, 99.9),
        )
else:
    print("Carregamento das imagens desativado por LOAD_IMAGE_DATA = False.")

effective_thresholds_df.head()

### 5. Baseline versus melhor threshold por exame

A tabela reúne o Dice e o threshold HU do baseline, a melhor alternativa do
mesmo exame e o respectivo delta. Delta positivo significa melhora sobre P99.9;
delta negativo significa que reduzir o percentil piorou o resultado.

In [ ]:
# Monta uma linha por exame com baseline, melhor alternativa e delta.
if effective_thresholds_df.empty:
    threshold_delta_details = largest_threshold_changes.copy()
    print("Ative LOAD_IMAGE_DATA para incluir os thresholds efetivos.")
else:
    thresholds_hu_wide = effective_thresholds_df.pivot(
        index="IMG_ID", columns="upper_percentile", values="max_threshold_hu"
    )
    threshold_delta_details = largest_threshold_changes.merge(
        thresholds_hu_wide,
        left_on="IMG_ID",
        right_index=True,
        how="left",
        validate="one_to_one",
    )
    threshold_delta_details["baseline_threshold_hu"] = threshold_delta_details[99.9]
    threshold_delta_details["best_threshold_hu"] = threshold_delta_details.apply(
        lambda row: row[99.7]
        if row["best_threshold_variant"] == "upper_p997"
        else row[99.5],
        axis=1,
    )

threshold_delta_details["best_threshold"] = threshold_delta_details[
    "best_threshold_variant"
].map(THRESHOLD_LABELS)
threshold_delta_details["result"] = "Empate"
threshold_delta_details.loc[threshold_delta_details["delta_dice"] > 0, "result"] = "Melhorou"
threshold_delta_details.loc[threshold_delta_details["delta_dice"] < 0, "result"] = "Piorou"

comparison_columns = [
    "IMG_ID", "baseline_dice", "baseline_threshold_hu", "best_threshold",
    "best_threshold_dice", "best_threshold_hu", "delta_dice", "result",
]
display(
    threshold_delta_details.reindex(columns=comparison_columns)
    .sort_values("delta_dice", ascending=False)
    .reset_index(drop=True)
    .round(3)
)

### 6. Configuração com maior Dice por exame

Conta quantos exames foram vencidos isoladamente por cada percentil. Quando
duas ou três configurações atingem exatamente o mesmo maior Dice, o exame é
registrado como empate.

In [ ]:
# Resume qual percentil alcança o maior Dice em cada exame, preservando empates.
maximum_dice = threshold_dice_wide.max(axis=1)
number_of_best_variants = threshold_dice_wide.eq(maximum_dice, axis=0).sum(axis=1)
best_variant_by_image = threshold_dice_wide.idxmax(axis=1).where(
    number_of_best_variants.eq(1), "tie"
)
best_threshold_counts = (
    best_variant_by_image.value_counts()
    .rename_axis("best_threshold")
    .reset_index(name="images")
)
best_threshold_counts["best_threshold"] = best_threshold_counts["best_threshold"].replace(
    {**THRESHOLD_LABELS, "tie": "Empate"}
)
best_threshold_counts["percent"] = 100 * best_threshold_counts["images"] / len(
    best_variant_by_image
)
best_threshold_counts.round(2)

### 7. Visualização das maiores variações

O gráfico à esquerda mostra o delta de Dice dos exames mais sensíveis. O
gráfico à direita liga o threshold HU do baseline ao threshold HU da melhor
alternativa, permitindo observar a intensidade da mudança aplicada.

In [ ]:
# Destaca visualmente as maiores melhorias e pioras em relação ao baseline.
plot_delta = threshold_delta_details.sort_values("delta_dice").copy()
colors = plot_delta["delta_dice"].map(
    lambda value: "#2a9d8f" if value > 0 else "#d1495b"
)
fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

axes[0].barh(plot_delta["IMG_ID"].astype(str), plot_delta["delta_dice"], color=colors)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("Delta Dice (melhor threshold - baseline)", fontsize=12)
axes[0].set_ylabel("Exame", fontsize=12)

if not effective_thresholds_df.empty:
    positions = range(len(plot_delta))
    axes[1].scatter(
        plot_delta["baseline_threshold_hu"], positions,
        color="#1f77b4", label="P99.9 baseline", s=38,
    )
    axes[1].scatter(
        plot_delta["best_threshold_hu"], positions,
        color="#e76f51", label="Melhor alternativa", s=38,
    )
    for position, (_, row) in zip(positions, plot_delta.iterrows()):
        axes[1].plot(
            [row["baseline_threshold_hu"], row["best_threshold_hu"]],
            [position, position], color="#888888", linewidth=0.8,
        )
    axes[1].set_yticks(list(positions), plot_delta["IMG_ID"].astype(str))
    axes[1].set_xlabel("Threshold superior efetivo (HU)", fontsize=12)
    axes[1].set_ylabel("Exame", fontsize=12)
    axes[1].legend()
else:
    axes[1].set_visible(False)

plt.show()

### 8. Melhor threshold dentro da faixa de intensidades da imagem

Para os exames com maior variação de Dice, esta tabela identifica qual entre
P99.9, P99.7 e P99.5 produziu o maior resultado. O threshold efetivo é situado
entre a menor e a maior intensidade do volume **após o mesmo downsampling do
pipeline**. A distância até o máximo ajuda a verificar se o percentil está
apenas removendo valores extremos ou recortando uma parcela mais ampla da faixa
superior de intensidades.

In [ ]:
# Relaciona o threshold vencedor à faixa HU real de cada volume analisado.
if effective_thresholds_df.empty:
    threshold_intensity_analysis = pd.DataFrame()
    print(
        "Ative LOAD_IMAGE_DATA para comparar o melhor threshold "
        "com as intensidades mínima e máxima."
    )
else:
    selected_ids = largest_threshold_changes["IMG_ID"].astype(int)
    selected_baseline_dice = threshold_dice_wide.loc[selected_ids, "baseline"]
    selected_winner_variant = best_variant_by_image.loc[selected_ids]
    selected_best_dice = maximum_dice.loc[selected_ids]

    winner_percentile = selected_winner_variant.map(
        {"baseline": 99.9, "upper_p997": 99.7, "upper_p995": 99.5}
    )

    threshold_intensity_analysis = pd.DataFrame(
        {
            "IMG_ID": selected_ids.to_numpy(),
            "best_threshold_variant": selected_winner_variant.to_numpy(),
            "best_upper_percentile": winner_percentile.to_numpy(),
            "baseline_dice": selected_baseline_dice.to_numpy(),
            "best_dice": selected_best_dice.to_numpy(),
        }
    )

    # Cada exame repete os mesmos limites nas três linhas de percentil.
    intensity_bounds = (
        effective_thresholds_df.groupby("IMG_ID", as_index=False)
        .agg(
            image_min_hu=("image_min_hu", "first"),
            image_max_hu=("image_max_hu", "first"),
            image_intensity_range_hu=("image_intensity_range_hu", "first"),
        )
    )
    threshold_intensity_analysis = threshold_intensity_analysis.merge(
        intensity_bounds,
        on="IMG_ID",
        how="left",
        validate="one_to_one",
    )

    # Anexa o valor HU correspondente ao percentil vencedor de cada exame.
    winner_thresholds = effective_thresholds_df.rename(
        columns={
            "upper_percentile": "best_upper_percentile",
            "max_threshold_hu": "best_threshold_hu",
        }
    )[["IMG_ID", "best_upper_percentile", "best_threshold_hu"]]
    threshold_intensity_analysis = threshold_intensity_analysis.merge(
        winner_thresholds,
        on=["IMG_ID", "best_upper_percentile"],
        how="left",
        validate="one_to_one",
    )

    # Mede o ganho do melhor percentil em relação ao baseline P99.9.
    threshold_intensity_analysis["delta_dice"] = (
        threshold_intensity_analysis["best_dice"]
        - threshold_intensity_analysis["baseline_dice"]
    )

    threshold_intensity_analysis["distance_threshold_to_max_hu"] = (
        threshold_intensity_analysis["image_max_hu"]
        - threshold_intensity_analysis["best_threshold_hu"]
    )
    threshold_intensity_analysis["threshold_position_percent"] = 100 * (
        threshold_intensity_analysis["best_threshold_hu"]
        - threshold_intensity_analysis["image_min_hu"]
    ) / threshold_intensity_analysis["image_intensity_range_hu"]
    threshold_intensity_analysis["best_threshold"] = (
        threshold_intensity_analysis["best_threshold_variant"]
        .replace({**THRESHOLD_LABELS, "tie": "Empate"})
    )

    analysis_columns = [
        "IMG_ID",
        "image_min_hu",
        "image_max_hu",
        "best_threshold",
        "best_threshold_hu",
        "distance_threshold_to_max_hu",
        "threshold_position_percent",
        "baseline_dice",
        "best_dice",
        "delta_dice",
    ]
    display(
        threshold_intensity_analysis[analysis_columns]
        .sort_values("best_dice", ascending=False)
        .reset_index(drop=True)
        .round(3)
    )

## Histogramas de casos representativos

São selecionados quatro exames: um em que P99.9 vence, um em que P99.7 vence,
um em que P99.5 vence e um empate. Dentro de cada grupo, escolhe-se o exame
cujo melhor Dice está mais próximo da mediana do próprio grupo, evitando usar
somente resultados extremos.

Nos gráficos, cada linha vertical de percentil representa o limite superior
real em HU daquele exame. Assim, P99.5 mantém os voxels até sua linha, P99.7
mantém também a faixa entre P99.5 e P99.7, e P99.9 preserva ainda a faixa entre
P99.7 e P99.9. Tudo acima de P99.9 é removido nas três configurações.

In [ ]:
HISTOGRAM_WINNER_ORDER = ["baseline", "upper_p997", "upper_p995", "tie"]
HISTOGRAM_WINNER_LABELS = {
    "baseline": "P99.9 vencedor",
    "upper_p997": "P99.7 vencedor",
    "upper_p995": "P99.5 vencedor",
    "tie": "Empate",
}

# Relaciona cada exame à configuração vencedora e ao maior Dice obtido.
winner_case_pool = pd.DataFrame(
    {
        "winner": best_variant_by_image,
        "winning_dice": threshold_dice_wide.max(axis=1),
    }
)
representative_rows = []

for winner in HISTOGRAM_WINNER_ORDER:
    winner_group = winner_case_pool.loc[winner_case_pool["winner"].eq(winner)].copy()
    if winner_group.empty:
        continue

    group_median = winner_group["winning_dice"].median()
    winner_group["distance_to_group_median"] = (
        winner_group["winning_dice"] - group_median
    ).abs()
    representative = winner_group.sort_values(
        ["distance_to_group_median", "winning_dice"],
        ascending=[True, False],
    ).iloc[0]
    image_id = int(representative.name)
    representative_rows.append(
        {
            "winner": winner,
            "case": HISTOGRAM_WINNER_LABELS[winner],
            "IMG_ID": image_id,
            "group_images": len(winner_group),
            "group_median_dice": group_median,
            "winning_dice": representative["winning_dice"],
            "dice_p999": threshold_dice_wide.loc[image_id, "baseline"],
            "dice_p997": threshold_dice_wide.loc[image_id, "upper_p997"],
            "dice_p995": threshold_dice_wide.loc[image_id, "upper_p995"],
        }
    )

representative_histogram_cases = pd.DataFrame(representative_rows)
representative_case_columns = [
    "case",
    "IMG_ID",
    "group_images",
    "group_median_dice",
    "winning_dice",
    "dice_p999",
    "dice_p997",
    "dice_p995",
]
display(representative_histogram_cases[representative_case_columns].round(4))

### Estatísticas de intensidade e desempenho por threshold

Os histogramas desta seção são calculados para **todas as imagens da validação**.
A tabela abaixo continua destacando os quatro exames representativos e é
agrupada hierarquicamente por caso e exame. Para cada imagem são mostradas a
média, a mediana e a intensidade máxima do histograma completo e da distribuição
restrita a intensidades maiores ou iguais ao limite HU configurado. Dentro de cada caso aparecem
P99.5, P99.7 e P99.9, seus valores efetivos, Dice e variação contra P99.9.

In [ ]:
cohort_histogram_summary_df = pd.DataFrame()
histogram_bins_df = pd.DataFrame()
cohort_effective_thresholds_df = pd.DataFrame()
histogram_summary_df = pd.DataFrame()
representative_effective_thresholds_df = pd.DataFrame()
representative_threshold_details = pd.DataFrame()

if not LOAD_IMAGE_DATA:
    print("Ative LOAD_IMAGE_DATA para calcular os histogramas.")
elif IMAGECAS_PATH is None:
    print("Defina IMAGECAS_BASE_PATH para calcular os histogramas.")
else:
    cohort_image_ids = threshold_dice_wide.index.astype(int).tolist()
    representative_ids = representative_histogram_cases["IMG_ID"].astype(int).tolist()

    # Uma única passagem por cada volume produz histogramas e os três percentis.
    cohort_histogram_summary_df, histogram_bins_df = (
        compute_intensity_histogram_analysis(
            cohort_image_ids,
            IMAGECAS_PATH,
            run_config["effective_base_config"],
            dense_min_hu=DENSE_INTENSITY_MIN_HU,
            bins=HISTOGRAM_BINS,
            percentiles=(99.5, 99.7, 99.9),
            progress_every=HISTOGRAM_PROGRESS_EVERY,
        )
    )

    percentile_columns = {99.5: "p995_hu", 99.7: "p997_hu", 99.9: "p999_hu"}
    cohort_effective_thresholds_df = pd.concat(
        [
            cohort_histogram_summary_df[["IMG_ID", column]].rename(
                columns={column: "max_threshold_hu"}
            ).assign(upper_percentile=percentile)
            for percentile, column in percentile_columns.items()
        ],
        ignore_index=True,
    )

    # Recorta a tabela de intensidade dos quatro casos usados na análise visual.
    histogram_summary_df = representative_histogram_cases.merge(
        cohort_histogram_summary_df.loc[
            cohort_histogram_summary_df["IMG_ID"].isin(representative_ids)
        ],
        on="IMG_ID",
        how="left",
        validate="one_to_one",
    )
    representative_effective_thresholds_df = cohort_effective_thresholds_df.loc[
        cohort_effective_thresholds_df["IMG_ID"].isin(representative_ids)
    ].copy()

    representative_threshold_details = threshold_performance_df.loc[
        threshold_performance_df["IMG_ID"].isin(representative_ids),
        ["IMG_ID", "variant", "upper_percentile", "dice_artery"],
    ].merge(
        representative_effective_thresholds_df[
            ["IMG_ID", "upper_percentile", "max_threshold_hu"]
        ],
        on=["IMG_ID", "upper_percentile"],
        how="left",
        validate="one_to_one",
    )
    representative_threshold_details = representative_threshold_details.merge(
        representative_histogram_cases[["IMG_ID", "case", "winner"]],
        on="IMG_ID",
        how="left",
        validate="many_to_one",
    )
    representative_threshold_details["threshold"] = (
        representative_threshold_details["variant"].map(THRESHOLD_LABELS)
    )

    baseline_details = representative_threshold_details.loc[
        representative_threshold_details["variant"].eq("baseline"),
        ["IMG_ID", "dice_artery", "max_threshold_hu"],
    ].rename(
        columns={
            "dice_artery": "baseline_dice",
            "max_threshold_hu": "baseline_threshold_hu",
        }
    )
    representative_threshold_details = representative_threshold_details.merge(
        baseline_details,
        on="IMG_ID",
        how="left",
        validate="many_to_one",
    )
    representative_threshold_details["delta_dice_vs_p999"] = (
        representative_threshold_details["dice_artery"]
        - representative_threshold_details["baseline_dice"]
    )
    representative_threshold_details["threshold_reduction_vs_p999_hu"] = (
        representative_threshold_details["baseline_threshold_hu"]
        - representative_threshold_details["max_threshold_hu"]
    )

In [ ]:
combined_threshold_statistics = pd.DataFrame()
if histogram_summary_df.empty or representative_threshold_details.empty:
    print("Não há estatísticas de intensidade e threshold para exibir.")
else:
    intensity_statistics = histogram_summary_df[
        [
            "IMG_ID",
            "full_mean_hu",
            "full_median_hu",
            "full_max_hu",
            "dense_voxel_percent",
            "dense_mean_hu",
            "dense_median_hu",
            "dense_max_hu",
        ]
    ]
    combined_threshold_statistics = representative_threshold_details.merge(
        intensity_statistics,
        on="IMG_ID",
        how="left",
        validate="many_to_one",
    )

    # Mantém a mesma ordem dos quatro casos escolhidos no notebook.
    case_order = representative_histogram_cases["case"].tolist()
    combined_threshold_statistics["case"] = pd.Categorical(
        combined_threshold_statistics["case"],
        categories=case_order,
        ordered=True,
    )
    combined_threshold_statistics = combined_threshold_statistics.sort_values(
        ["case", "max_threshold_hu"],
        ascending=[True, False],
    )

    table_columns = {
        "case": "Caso",
        "IMG_ID": "IMG_ID",
        "threshold": "Threshold",
        "max_threshold_hu": "Threshold efetivo (HU)",
        "dice_artery": "Dice",
        "delta_dice_vs_p999": "Delta Dice vs P99.9",
        "threshold_reduction_vs_p999_hu": "Redução vs P99.9 (HU)",
        "full_mean_hu": "Média HU - completo",
        "full_median_hu": "Mediana HU - completo",
        "full_max_hu": "Máximo HU - completo",
        "dense_voxel_percent": (
            f"Voxels >= {DENSE_INTENSITY_MIN_HU:g} HU (%)"
        ),
        "dense_mean_hu": f"Média HU - >= {DENSE_INTENSITY_MIN_HU:g}",
        "dense_median_hu": f"Mediana HU - >= {DENSE_INTENSITY_MIN_HU:g}",
        "dense_max_hu": f"Máximo HU - >= {DENSE_INTENSITY_MIN_HU:g}",
    }
    combined_threshold_statistics = (
        combined_threshold_statistics[list(table_columns)]
        .rename(columns=table_columns)
        .set_index(["Caso", "IMG_ID", "Threshold"])
    )
    display(combined_threshold_statistics.round(3))

### Distribuições médias de toda a validação

A primeira linha usa **todos os exames da validação**. As demais linhas mostram
separadamente as médias dos exames em que P99.9 venceu, P99.7 venceu, P99.5
venceu e os thresholds empataram. Em cada linha são apresentados o histograma
completo e a cauda acima do limite HU configurado. Cada exame é normalizado antes da média e
todos são reamostrados na mesma grade HU; assim, volumes maiores não dominam o
resultado. As linhas verticais são os thresholds HU médios do respectivo grupo.

In [ ]:
normalized_cohort_histograms = pd.DataFrame()
if histogram_bins_df.empty:
    print("Não há histogramas calculados para gerar as médias da coorte.")
else:
    cohort_image_ids = threshold_dice_wide.index.astype(int).tolist()
    normalized_cohort_histograms = pd.concat(
        [
            build_normalized_intensity_histograms(
                histogram_bins_df,
                image_ids=cohort_image_ids,
                histogram_name=histogram_name,
                bins=HISTOGRAM_BINS,
            )
            for histogram_name in ("full", "dense_hu")
        ],
        ignore_index=True,
    )

    winner_groups = (
        best_variant_by_image.rename("winner")
        .rename_axis("IMG_ID")
        .reset_index()
    )
    winner_groups["IMG_ID"] = winner_groups["IMG_ID"].astype(int)
    group_specs = [
        ("all", "Todos os exames", cohort_image_ids),
        (
            "baseline",
            "Casos em que P99.9 venceu",
            winner_groups.loc[winner_groups["winner"].eq("baseline"), "IMG_ID"].tolist(),
        ),
        (
            "upper_p997",
            "Casos em que P99.7 venceu",
            winner_groups.loc[winner_groups["winner"].eq("upper_p997"), "IMG_ID"].tolist(),
        ),
        (
            "upper_p995",
            "Casos em que P99.5 venceu",
            winner_groups.loc[winner_groups["winner"].eq("upper_p995"), "IMG_ID"].tolist(),
        ),
        (
            "tie",
            "Casos com empate",
            winner_groups.loc[winner_groups["winner"].eq("tie"), "IMG_ID"].tolist(),
        ),
    ]
    threshold_colors = {99.5: "#f4a261", 99.7: "#7b2cbf", 99.9: "#d1495b"}

    fig, axes = plt.subplots(
        len(group_specs),
        2,
        figsize=(16, 4.0 * len(group_specs)),
        constrained_layout=True,
        squeeze=False,
    )

    for row_index, (_, group_label, group_image_ids) in enumerate(group_specs):
        if not group_image_ids:
            axes[row_index, 0].set_visible(False)
            axes[row_index, 1].set_visible(False)
            continue

        group_profiles = normalized_cohort_histograms.loc[
            normalized_cohort_histograms["IMG_ID"].isin(group_image_ids)
        ]
        group_thresholds = (
            cohort_effective_thresholds_df.loc[
                cohort_effective_thresholds_df["IMG_ID"].isin(group_image_ids)
            ]
            .groupby("upper_percentile")["max_threshold_hu"]
            .mean()
            .sort_index()
        )

        for column_index, (histogram_name, title) in enumerate(
            (
                ("full", "Distribuição completa"),
                (
                    "dense_hu",
                    f"Intensidades >= {DENSE_INTENSITY_MIN_HU:g} HU",
                ),
            )
        ):
            ax = axes[row_index, column_index]
            profiles = group_profiles.loc[
                group_profiles["histogram"].eq(histogram_name)
            ]
            histogram = build_mean_normalized_intensity_histogram(
                profiles,
            )
            group_mean_hu, group_median_hu = (
                calculate_binned_intensity_mean_median(
                    histogram,
                    value_column="mean_probability",
                )
            )
            lower = (
                histogram["mean_probability"] - histogram["std_probability"]
            ).clip(lower=0)
            upper = histogram["mean_probability"] + histogram["std_probability"]
            ax.fill_between(
                histogram["bin_center_hu"],
                lower,
                upper,
                color="#31688e",
                alpha=0.18,
                label="Média ±1 desvio-padrão",
            )
            ax.plot(
                histogram["bin_center_hu"],
                histogram["mean_probability"],
                color="#111111",
                linewidth=2.3,
                label="Probabilidade média",
            )
            ax.axvline(
                group_mean_hu,
                color="#2a9d8f",
                linestyle=":",
                linewidth=1.4,
                label=f"Média: {group_mean_hu:.1f} HU",
            )
            ax.axvline(
                group_median_hu,
                color="#555555",
                linestyle="--",
                linewidth=1.3,
                label=f"Mediana: {group_median_hu:.1f} HU",
            )

            p995, p997, p999 = (
                float(group_thresholds.loc[percentile])
                for percentile in (99.5, 99.7, 99.9)
            )
            histogram_max = float(histogram["bin_right_hu"].max())
            ax.axvspan(p995, p997, color="#f4a261", alpha=0.12)
            ax.axvspan(p997, p999, color="#d1495b", alpha=0.10)
            ax.axvspan(p999, histogram_max, color="#555555", alpha=0.08)
            for percentile in (99.5, 99.7, 99.9):
                ax.axvline(
                    group_thresholds.loc[percentile],
                    color=threshold_colors[percentile],
                    linewidth=1.8,
                    linestyle="--",
                    label=(
                        f"P{percentile:g} médio: "
                        f"{group_thresholds.loc[percentile]:.0f} HU"
                    ),
                )

            if histogram_name == "full":
                ax.set_yscale("log")
            ax.set_title(f"{group_label} (n={len(group_image_ids)}) | {title}")
            ax.set_xlabel("Intensidade (HU)")
            ax.set_ylabel("Probabilidade média por bin")
            ax.legend(fontsize=10, loc="best")

    plt.show()